# 10. Double Booking: Handling Duplicate Data

## Introduction

Sometimes a dataset looks perfectly fine at a glance, but secretly contains duplicate rows. This happens for several reasons—perhaps an API glitched, a database query pulled duplicate records, or someone manually copy-pasted entries incorrectly.

Regardless of how they get there, running analysis or machine learning models on duplicated data leads to skewed statistics and flawed results.

Luckily, Pandas makes finding and removing duplicate data simple!

---

## 1. Finding Duplicate Data with `.duplicated()`

To check for duplicate rows, use the `.duplicated()` method:

```python
df.duplicated()

```

This returns a boolean Series (`True` or `False` for each row). A value of `True` indicates that the row is an exact duplicate of a previous row:

```text
0    False
1    False
2     True
3    False
dtype: bool

```

### Viewing the Duplicate Rows

To inspect the actual duplicated rows rather than just seeing `True`/`False` flags, use the Series as a filter mask on your DataFrame:

```python
# Returns only the rows marked as True (duplicates)
duplicate_rows = df[df.duplicated()]

```

---

## 2. Removing Duplicates with `.drop_duplicates()`

Once you've identified duplicates, you can clean them up. The standard approach is `.drop_duplicates()`, which removes rows that are **100% identical** across every single column:

```python
no_dupes = df.drop_duplicates()

```

---

### Exact Duplicates vs. Subset Duplicates

In real-world data, rows often represent the same underlying entity without being 100% identical in every field.

> * **Example:** Imagine a library catalog where the same book was entered twice with slight variations in author formatting:
> * **Row A:** *The Lord of the Rings* | Author: `JRR Tolkien`
> * **Row B:** *The Lord of the Rings* | Author: `J.R.R. Tolkien`
> 
> 

Running `.drop_duplicates()` on the entire DataFrame will **not** catch this because the author strings don't match.

To fix this, target a column that uniquely identifies each record (such as an `ISBN` or `ticket_id`) using the `subset` parameter:

```python
# Keep the first instance and drop subsequent duplicates based on ISBN
unique_books = catalog.drop_duplicates(subset=['ISBN'])

```

When using `subset`, Pandas keeps the **first occurrence** of each unique value by default and discards any subsequent duplicates.

---

## Exercise: Double Booking

> ✈️ **The Scenario:** You've been given a dataset of flight ticket bookings containing duplicate entries. Your task is to create a clean DataFrame with exactly **one row per unique ticket booking**.

### Your Tasks

1. **Test basic deduplication:** Call `.drop_duplicates()` on the bookings DataFrame without any parameters. Notice that some duplicate bookings still remain. Add a comment in your code explaining **why** full-row deduplication failed to remove them.
2. **Apply targeted deduplication:** Call `.drop_duplicates()` using the `subset` parameter on the column that uniquely identifies a booking (e.g., `booking_id` or `ticket_number`) to cleanly strip out all remaining duplicate tickets.

In [4]:
import pandas as pd

data = {
  'ticket_id': [
    'TKT-88421', 'TKT-88421', 'TKT-99213',
    'TKT-55310', 'TKT-55310', 'TKT-66772',
    'TKT-11890', 'TKT-11890', 'TKT-77445',
    'TKT-33109', 'TKT-18479', 'TKT-90002'
  ],
  'passenger_name': [
    'Don Draper', 'Don Draper', 'Emily Davis',
    'Roger Sterling', 'Roger Sterling', 'Harper Stern',
    'Pete Campbell', 'Pete Campbell', 'Robert Spearing',
    'Olivia Martinez', 'Andrew Largeman', 'Kendall Roy'
  ],
  'flight': [
    'PA102', 'PA102', 'DL404',
    'UA220', 'UA220', 'UA178',
    'AA778', 'AA778', 'BA305',
    'DL909', 'UA347', 'PRV-JET'
  ],
  'route': [
    'LGW → LAX', 'LGA → LAX', 'ATL → SEA',
    'JFK → CDG', 'JFK → CDG', 'JFK → LHR',
    'MCI → LGA', 'MCI → LGA', 'LHR → SFO',
    'DEN → PHX', 'LAX → EWR', 'NYC → Oslo'
  ]
}

tickets = pd.DataFrame(data)
tickets

,ticket_id,passenger_name,flight,route
0,TKT-88421,Don Draper,PA102,LGW → LAX
1,TKT-88421,Don Draper,PA102,LGA → LAX
2,TKT-99213,Emily Davis,DL404,ATL → SEA
3,TKT-55310,Roger Sterling,UA220,JFK → CDG
4,TKT-55310,Roger Sterling,UA220,JFK → CDG
5,TKT-66772,Harper Stern,UA178,JFK → LHR
6,TKT-11890,Pete Campbell,AA778,MCI → LGA
7,TKT-11890,Pete Campbell,AA778,MCI → LGA
8,TKT-77445,Robert Spearing,BA305,LHR → SFO
9,TKT-33109,Olivia Martinez,DL909,DEN → PHX


In [5]:
duplicates = tickets[tickets.duplicated()]
print(duplicates)


   ticket_id  passenger_name flight      route
4  TKT-55310  Roger Sterling  UA220  JFK → CDG
7  TKT-11890   Pete Campbell  AA778  MCI → LGA


In [7]:
tickets_clean = tickets.drop_duplicates()
# Some duplicates are not exactly the same and differ in some columns, and are not getting dropped
tickets_clean

,ticket_id,passenger_name,flight,route
0,TKT-88421,Don Draper,PA102,LGW → LAX
1,TKT-88421,Don Draper,PA102,LGA → LAX
2,TKT-99213,Emily Davis,DL404,ATL → SEA
3,TKT-55310,Roger Sterling,UA220,JFK → CDG
5,TKT-66772,Harper Stern,UA178,JFK → LHR
6,TKT-11890,Pete Campbell,AA778,MCI → LGA
8,TKT-77445,Robert Spearing,BA305,LHR → SFO
9,TKT-33109,Olivia Martinez,DL909,DEN → PHX
10,TKT-18479,Andrew Largeman,UA347,LAX → EWR
11,TKT-90002,Kendall Roy,PRV-JET,NYC → Oslo


In [8]:
tickets_unique = tickets.drop_duplicates(subset=['ticket_id'])
tickets_unique

,ticket_id,passenger_name,flight,route
0,TKT-88421,Don Draper,PA102,LGW → LAX
2,TKT-99213,Emily Davis,DL404,ATL → SEA
3,TKT-55310,Roger Sterling,UA220,JFK → CDG
5,TKT-66772,Harper Stern,UA178,JFK → LHR
6,TKT-11890,Pete Campbell,AA778,MCI → LGA
8,TKT-77445,Robert Spearing,BA305,LHR → SFO
9,TKT-33109,Olivia Martinez,DL909,DEN → PHX
10,TKT-18479,Andrew Largeman,UA347,LAX → EWR
11,TKT-90002,Kendall Roy,PRV-JET,NYC → Oslo
